# Lesson 1 — Tensor & Shape Thinking

## 学习目标

这一节的重点不是记住 PyTorch API，而是建立 Transformer 中非常重要的 **Tensor Shape Thinking**。

完成本节后，应能够：

1. 理解 Tensor 的 shape、ndim、dtype、device 和 numel；
2. 理解 `(B, T, D)` 在语言模型中的含义；
3. 根据 indexing / slicing 推导输出 shape；
4. 区分整数索引和 slice；
5. 理解 reshape、transpose、permute；
6. 看懂 Attention 中 `(B, T, D) -> (B, H, T, Dh)` 的变化。

以后看到 Tensor 时，第一反应应该是：

> 每一个维度代表什么？


In [1]:
import torch

x = torch.randn(2, 3, 4)

print("shape :", x.shape)
print("ndim  :", x.ndim)
print("dtype :", x.dtype)
print("device:", x.device)
print("numel :", x.numel())

shape : torch.Size([2, 3, 4])
ndim  : 3
dtype : torch.float32
device: cpu
numel : 24


## 1. Tensor Shape 的语义

假设：

$$
x \in \mathbb{R}^{B \times T \times D}
$$

在语言模型中通常记作：

$$
x.shape = (B, T, D)
$$

其中：

- $B$：Batch Size
- $T$：Sequence Length
- $D$：Model Dimension

例如：

$$
x.shape = (2, 3, 4)
$$

可以理解为：

- 2 个样本；
- 每个样本有 3 个 token；
- 每个 token 用一个 4 维向量表示。

因此不要只把 `(2, 3, 4)` 看成三个数字。

应该逐渐形成：

$$
(B, T, D)
=
(\text{batch}, \text{token}, \text{feature})
$$


In [ ]:
B = 2
T = 3
D = 4

x = torch.randn(B, T, D)

print("x.shape =", x.shape)

print("Batch 0 shape:", x[0].shape)
print("Token 0 shape:", x[0, 0].shape)

x.shape = torch.Size([2, 3, 4])
Batch 0 shape: torch.Size([3, 4])
Token 0 shape: torch.Size([4])


## 2. Indexing 与 Slicing

这里先记住最重要的规则：

> 整数索引会消掉一个维度，slice 会保留这个维度。

例如原始 Tensor：

$$
x.shape = (2, 3, 4)
$$

### 整数索引

`x[0]`

第 0 维使用整数索引，因此第 0 维消失：

$$
(2, 3, 4)
\rightarrow
(3, 4)
$$

### Slice

`x[0:1]`

`0:1` 是 slice，所以这个维度仍然保留：

$$
(2, 3, 4)
\rightarrow
(1, 3, 4)
$$

这是之后进行 Tensor shape 推导时非常重要的区别。


In [3]:
x = torch.randn(2, 3, 4)

print("Original :", x.shape)

print("x[0]     :", x[0].shape)
print("x[0:1]   :", x[0:1].shape)

print("x[:, 0]  :", x[:, 0].shape)
print("x[:, 0:1]:", x[:, 0:1].shape)

print("x[:, :, 0]  :", x[:, :, 0].shape)
print("x[:, :, 0:1]:", x[:, :, 0:1].shape)


Original : torch.Size([2, 3, 4])
x[0]     : torch.Size([3, 4])
x[0:1]   : torch.Size([1, 3, 4])
x[:, 0]  : torch.Size([2, 4])
x[:, 0:1]: torch.Size([2, 1, 4])
x[:, :, 0]  : torch.Size([2, 3])
x[:, :, 0:1]: torch.Size([2, 3, 1])


## 3. 逐维记账法

以后不要靠感觉猜 shape，而要逐个维度分析。

例如：

`x[:, 0, :]`

原始：

$$
(B, T, D) = (2, 3, 4)
$$

逐维分析：

- B 维使用 `:`：保留，大小为 2
- T 维使用整数 `0`：维度消失
- D 维使用 `:`：保留，大小为 4

所以：

$$
x[:, 0, :].shape = (2, 4)
$$

再看：

`x[:, 0:1, :]`

- B：保留 2
- T：使用 slice，维度保留，但长度变成 1
- D：保留 4

所以：

$$
x[:, 0:1, :].shape = (2, 1, 4)
$$

可以形成简单规则：

- `0`、`1`、`2`：整数索引，维度消失
- `:`、`0:1`、`1:`、`1:3`：slice，维度保留


## 4. Reshape

`reshape` 的作用是重新组织 Tensor 的维度。

重要规则：

> reshape 前后的元素总数必须相同。

例如：

$$
2 \times 3 \times 4 = 24
$$

所以 `(2, 3, 4)` 可以 reshape 成：

$$
(6,4)
$$

因为：

$$
6 \times 4 = 24
$$

也可以变成：

$$
(2,12)
$$

因为：

$$
2 \times 12 = 24
$$

注意：

Indexing 不会自动 flatten。

只有显式调用 reshape 等操作时，维度才会真正被合并。


In [4]:
x = torch.arange(24).reshape(2, 3, 4)

print("Original:", x.shape)

a = x.reshape(6, 4)
b = x.reshape(2, 12)
c = x.reshape(-1, 4)

print("a:", a.shape)
print("b:", b.shape)
print("c:", c.shape)


Original: torch.Size([2, 3, 4])
a: torch.Size([6, 4])
b: torch.Size([2, 12])
c: torch.Size([6, 4])


### `-1` 的作用

PyTorch 允许使用 `-1` 自动推导某个维度。

例如：

`x.reshape(-1, 4)`

因为总元素数量为：

$$
24
$$

所以 PyTorch 自动求：

$$
? \times 4 = 24
$$

因此：

$$
? = 6
$$

最终：

$$
(2,3,4)
\rightarrow
(6,4)
$$


## 5. Transpose

`transpose(dim0, dim1)` 的作用是交换两个维度。

假设：

$$
x.shape = (2,3,4)
$$

维度编号为：

$$
dim_0 = 2,\quad dim_1 = 3,\quad dim_2 = 4
$$

执行：

`x.transpose(1, 2)`

交换 dim 1 和 dim 2：

$$
(2,3,4)
\rightarrow
(2,4,3)
$$

注意：

> transpose 不是矩阵乘法，它只是交换 Tensor 的维度。


In [5]:
x = torch.randn(2, 3, 4)

y = x.transpose(1, 2)

print("x:", x.shape)
print("y:", y.shape)


x: torch.Size([2, 3, 4])
y: torch.Size([2, 4, 3])


## 6. Permute

`permute` 可以任意重新排列多个维度。

例如：

$$
x.shape = (2,3,4)
$$

执行：

`x.permute(2, 0, 1)`

注意 `(2, 0, 1)` 不是新的 shape。

它表示：

- 新 dim 0 = 原 dim 2
- 新 dim 1 = 原 dim 0
- 新 dim 2 = 原 dim 1

因此：

$$
(2,3,4)
\rightarrow
(4,2,3)
$$

可以理解为：

原顺序：

$$
(dim_0, dim_1, dim_2)
$$

变成：

$$
(dim_2, dim_0, dim_1)
$$


In [6]:
x = torch.randn(2, 3, 4)

y = x.permute(2, 0, 1)

print("x:", x.shape)
print("y:", y.shape)


x: torch.Size([2, 3, 4])
y: torch.Size([4, 2, 3])


## 7. Multi-Head Attention 中的 Shape

现在第一次把前面的知识放到 Transformer 中。

输入通常为：

$$
x.shape = (B,T,D)
$$

Multi-Head Attention 将模型维度 $D$ 拆成：

$$
D = H \times D_h
$$

其中：

- $H$：Number of Attention Heads
- $D_h$：Head Dimension

例如：

$$
B=2,\quad T=3,\quad D=8,\quad H=2
$$

那么：

$$
D_h = \frac{D}{H} = 4
$$

首先：

$$
(B,T,D)
\rightarrow
(B,T,H,D_h)
$$

也就是：

$$
(2,3,8)
\rightarrow
(2,3,2,4)
$$

然后交换 token 和 head 两个维度：

$$
(B,T,H,D_h)
\rightarrow
(B,H,T,D_h)
$$

最终：

$$
(2,2,3,4)
$$

这是以后 Q、K、V 最常见的 shape。


In [7]:
B = 2
T = 3
D = 8
H = 2

Dh = D // H

x = torch.randn(B, T, D)

q = x.reshape(B, T, H, Dh)
q = q.transpose(1, 2)

print("Input :", x.shape)
print("Output:", q.shape)


Input : torch.Size([2, 3, 8])
Output: torch.Size([2, 2, 3, 4])


## 8. Contiguous：暂时只需要认识

transpose 和 permute 通常不会真正重新排列底层的数据。

它们更多是在改变：

- shape
- stride
- Tensor 对底层内存的解释方式

因此 transpose 后的 Tensor 可能不是 contiguous。

现在只需要记住：

> Tensor 的逻辑 shape 和底层内存布局不是一回事。

后面的 CS336 Systems / Triton 部分会重新深入学习：

- contiguous memory
- stride
- memory access
- data movement


In [8]:
x = torch.randn(2, 3, 4)

y = x.transpose(1, 2)

print("x contiguous:", x.is_contiguous())
print("y contiguous:", y.is_contiguous())

z = y.contiguous()

print("z contiguous:", z.is_contiguous())


x contiguous: True
y contiguous: False
z contiguous: True


## 本节总结

### 1. Shape 要带语义

Transformer 中最常见：

$$
(B,T,D)
$$

分别表示：

- Batch
- Token / Sequence
- Model Dimension

### 2. 整数索引删除维度

例如：

`x[:, 0, :]`

$$
(B,T,D)
\rightarrow
(B,D)
$$

### 3. Slice 保留维度

例如：

`x[:, 0:1, :]`

$$
(B,T,D)
\rightarrow
(B,1,D)
$$

### 4. Reshape 改变维度分组

例如：

$$
(2,3,4)
\rightarrow
(6,4)
$$

元素总数保持不变。

### 5. Transpose / Permute 改变维度顺序

reshape 和 transpose 是完全不同的操作。

### 6. Attention 中的重要 Shape

$$
(B,T,D)
$$

经过 reshape：

$$
(B,T,H,D_h)
$$

再 transpose：

$$
(B,H,T,D_h)
$$

并满足：

$$
D = H D_h
$$
